In [1]:
import pandas as pd
import os
import json
import numpy as np

In [2]:
def parse_result_as_df_recommended(result_file):
    with open(result_file, 'r') as f:
        lines = f.readlines()
    json_data = json.loads(('').join(lines))
    
    # Handle evaluationResult array by extracting comprehensive statistics
    if 'evaluationResult' in json_data and isinstance(json_data['evaluationResult'], list):
        eval_results = json_data['evaluationResult']
        
        # Basic statistics
        json_data['evaluationResult_count'] = len(eval_results)

        # Exact matching statistics
        exact_matches = [item for item in eval_results if item.get('doesMatchExactly', False)]
        json_data['evaluationResult_exact_matches'] = len(exact_matches) > 0
        # json_data['evaluationResult_exact_rate'] = len(exact_matches) / len(eval_results) if eval_results else 0
        
        # Syntactic matching statistics
        syntactic_matches = [item for item in eval_results if item.get('doesMatchSyntactically', False)]
        json_data['evaluationResult_syntactic_matches'] = len(syntactic_matches) > 0
        # json_data['evaluationResult_syntactic_rate'] = len(syntactic_matches) / len(eval_results) if eval_results else 0
        
        # Semantic matching statistics
        semantic_matches = [item for item in eval_results if item.get('doesMatchSemantically', False)]
        json_data['evaluationResult_semantic_matches'] = len(semantic_matches) > 0
        # json_data['evaluationResult_semantic_rate'] = len(semantic_matches) / len(eval_results) if eval_results else 0
        
        # # Top suggestions
        # json_data['evaluationResult_top_suggestion'] = eval_results[0]['suggestion'] if eval_results else None
        # json_data['evaluationResult_top_rank'] = eval_results[0]['rank'] if eval_results else None
        
        # # Average rank of matches
        # if syntactic_matches:
        #     json_data['evaluationResult_avg_syntactic_rank'] = np.mean([item['rank'] for item in syntactic_matches])
        # else:
        #     json_data['evaluationResult_avg_syntactic_rank'] = None
            
        # Remove the original array to avoid DataFrame creation issues
        del json_data['evaluationResult']

        # Delete any key that starts with 'suggestionIn' to avoid redundancy
        keys_to_delete = [key for key in json_data.keys() if key.startswith('suggestionIn')]
        for key in keys_to_delete:
            del json_data[key]
    
    # Handle any other arrays in similar fashion
    for key, value in list(json_data.items()):
        if isinstance(value, list) and key != 'evaluationResult':  # Handle other potential arrays
            json_data[f'{key}_array_length'] = len(value)
            json_data[f'{key}_as_string'] = json.dumps(value)  # Keep as JSON string if needed
            del json_data[key]  # Remove original array
    
    df = pd.DataFrame(json_data, index=[0])
    return df

In [3]:
# Updated data loading code with proper JSON array handling
FORMULA_TEST_RESULT_DIR = '../test-results/formula/multi_term/'

formula_results_df = pd.DataFrame()

print("Loading formula results...")
for root, subdirs, files in os.walk(FORMULA_TEST_RESULT_DIR):
    for file in files:
        if file.endswith('.result.json'):
            try:
                df = parse_result_as_df_recommended(os.path.join(root, file))
                formula_results_df = pd.concat([formula_results_df, df], axis=0, ignore_index=True)
            except Exception as e:
                print(f"Error processing {file}: {e}")

print(f"Formula results loaded: {len(formula_results_df)} records")

Loading formula results...
Formula results loaded: 625 records


In [4]:
formula_results_df['suggestion_state'] = formula_results_df.apply(
    lambda row: 'no-suggestion' if not row['completionGenerated'] else row['suggestionExists'], axis=1)

In [5]:
formula_results_df[formula_results_df['modelName'] == 'classroom-fol'].sort_values('offset')

,modelName,filename,offset,term,line,character,incompletionLine,completionList,completionGenerated,suggestionExists,expectedCompletionWord,expectedCompletionLine,elapsedTimeInMs,evaluationResult_count,evaluationResult_exact_matches,evaluationResult_syntactic_matches,evaluationResult_semantic_matches,suggestion_state
297,classroom-fol,completion/classroom-fol-sig.als,107,->,9,20,Groups : Person ->,"Class, Group, Person, Student, Teacher, Int, S...",True,True,Group,Group,14.832791,9,True,True,False,True
304,classroom-fol,completion/classroom-fol-sig.als,136,extends,12,20,sig Teacher extends,"Class, Group, Person, Student, Int, String, no...",True,True,Person,Person {},8.285292,8,True,True,False,True
290,classroom-fol,completion/classroom-fol-sig.als,162,in,14,15,sig Student in,"Class, Group, Person, Teacher, Int, String, no...",True,True,Person,Person {},7.412667,8,True,True,False,True
310,classroom-fol,completion/classroom-fol-complete.als,207,in,17,22,all p: Person | p in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Student,Student,74.129000,190,True,False,True,True
313,classroom-fol,completion/classroom-fol-complete.als,256,in,21,26,all p: Person | p not in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Teacher,Teacher,28.437416,190,True,False,True,True
308,classroom-fol,completion/classroom-fol-complete.als,301,in,25,22,all p: Person | p in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Student,Student implies p not in Teacher,27.934333,190,True,False,True,True
312,classroom-fol,completion/classroom-fol-complete.als,326,in,25,47,all p: Person | p in Student implies p not in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Teacher,Teacher,45.473875,190,True,False,True,True
288,classroom-fol,completion/classroom-fol-complete.als,371,in,29,22,all p: Person | p in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Student,Student or p in Teacher,28.795459,190,True,False,True,True
300,classroom-fol,completion/classroom-fol-complete.als,387,in,29,38,all p: Person | p in Student or p in,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,True,Teacher,Teacher,25.075333,190,True,False,True,True
292,classroom-fol,completion/classroom-fol-complete.als,424,.,33,14,some Teacher.,"*Tutors.Person, *Tutors.Student, *Tutors.Teach...",True,True,Teaches,Teaches,20.734792,28,True,True,True,True


In [6]:
# failed_completions = formula_results_df[(formula_results_df['suggestion_state'].isin([False, 'no-suggestion'])) & (~formula_results_df['expectedCompletionLine'].str.contains('\('))]
failed_completions = formula_results_df[(formula_results_df['suggestion_state'].isin([False, 'no-suggestion']))]
failed_completions.count()

modelName                             48
filename                              48
offset                                48
term                                  48
line                                  48
character                             48
incompletionLine                      48
completionList                        48
completionGenerated                   48
suggestionExists                      48
expectedCompletionWord                48
expectedCompletionLine                48
elapsedTimeInMs                       48
evaluationResult_count                48
evaluationResult_exact_matches        48
evaluationResult_syntactic_matches    48
evaluationResult_semantic_matches     48
suggestion_state                      48
dtype: int64

In [7]:
failed_completions[failed_completions['modelName'] == 'classroom']

,modelName,filename,offset,term,line,character,incompletionLine,completionList,completionGenerated,suggestionExists,expectedCompletionWord,expectedCompletionLine,elapsedTimeInMs,evaluationResult_count,evaluationResult_exact_matches,evaluationResult_syntactic_matches,evaluationResult_semantic_matches,suggestion_state
361,classroom,completion/classroom-complete.als,962,&,73,32,all p:Person | some Teacher &,"*Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors...",True,False,,(^Tutors).p,34.442208,184,False,False,True,False
365,classroom,completion/classroom-complete.als,620,.,53,33,"all c:Class,s:Student | some s.","*Tutors.s, Teaches.c, Tutors.s, ^Tutors.s, ~Tu...",True,False,,(c.Groups),11.353625,33,False,False,False,False
374,classroom,completion/classroom-complete.als,863,in,69,30,"all c:Class,p:Person | p in","*Tutors.Teaches.c, *Tutors.p, *Tutors.p.*Tutor...",True,False,,(c.Groups).Group implies Teaches.c -> p in Tutors,27.081084,204,False,False,True,False


In [8]:
from IPython.display import HTML, display
# Set CSS to make figures use more width
display(HTML("""
<style>
    .output_png {
        display: block;
        margin: 0 auto;
        max-width: 100% !important;
        width: 100% !important;
    }
    .jp-OutputArea-output {
        overflow-x: auto;
    }
</style>
"""))

# Configure matplotlib backend for better notebook integration
%matplotlib inline
import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 0

In [9]:
# Show only the informative columns
failed_completions = failed_completions[['modelName', 'line', 'incompletionLine', 'term', 'expectedCompletionWord', 'expectedCompletionLine', 'completionList']].sort_values(['modelName', 'line'])
display(HTML(failed_completions.to_html(index=False)))

modelName,line,incompletionLine,term,expectedCompletionWord,expectedCompletionLine,completionList
classroom,53,"all c:Class,s:Student | some s.",.,,(c.Groups),"*Tutors.s, Teaches.c, Tutors.s, ^Tutors.s, ~Tutors.s, *Tutors.Person, *Tutors.Student, *Tutors.Teacher, Teaches.Class, Tutors.Person, Tutors.Student, Tutors.Teacher, ^Tutors.Person, ^Tutors.Student, ^Tutors.Teacher, ~Tutors.Person, ~Tutors.Student, ~Tutors.Teacher, *Tutors.Teaches, Teaches, Teaches.Groups, Teaches.univ, Tutors, Tutors.Teaches, Tutors.univ, ^Tutors.Teaches, ~Tutors.Teaches, ~Tutors, ~Tutors.univ, ^Tutors, ^Tutors.univ, *Tutors, *Tutors.univ"
classroom,69,"all c:Class,p:Person | p in",in,,(c.Groups).Group implies Teaches.c -> p in Tutors,"*Tutors.Teaches.c, *Tutors.p, *Tutors.p.*Tutors, *Tutors.p.Tutors, *Tutors.p.^Tutors, *Tutors.p.~Tutors, Teaches.c, Teaches.c.~Teaches, Tutors.Teaches.c, Tutors.p, Tutors.p.*Tutors, Tutors.p.Tutors, Tutors.p.^Tutors, Tutors.p.~Tutors, ^Tutors.Teaches.c, ^Tutors.p, ^Tutors.p.*Tutors, ^Tutors.p.Tutors, ^Tutors.p.^Tutors, ^Tutors.p.~Tutors, c, c.Groups.Group, c.Groups.univ, c.~Teaches, c.~Teaches.*Tutors, c.~Teaches.Tutors, c.~Teaches.^Tutors, c.~Teaches.~Tutors, p, p.*Tutors, p.Tutors, p.^Tutors, p.~Tutors, ~Teaches.p.*Tutors, ~Teaches.p.Teaches, ~Teaches.p.Tutors, ~Teaches.p.^Tutors, ~Teaches.p.~Tutors, ~Tutors.Teaches.c, ~Tutors.p, ~Tutors.p.*Tutors, ~Tutors.p.Tutors, ~Tutors.p.^Tutors, ~Tutors.p.~Tutors, *Tutors.Person, *Tutors.Person.*Tutors, *Tutors.Person.Tutors, *Tutors.Person.^Tutors, *Tutors.Person.~Tutors, *Tutors.Student, *Tutors.Student.*Tutors, *Tutors.Student.Tutors, *Tutors.Student.^Tutors, *Tutors.Student.~Tutors, *Tutors.Teacher, *Tutors.Teacher.*Tutors, *Tutors.Teacher.Tutors, *Tutors.Teacher.^Tutors, *Tutors.Teacher.~Tutors, *Tutors.Teaches.Class, Class, Class.Groups.Group, Class.Groups.univ, Class.~Teaches, Class.~Teaches.*Tutors, Class.~Teaches.Tutors, Class.~Teaches.^Tutors, Class.~Teaches.~Tutors, Group, Person, Person.*Tutors, Person.Tutors, Person.^Tutors, Person.~Tutors, Student, Student.*Tutors, Student.Tutors, Student.^Tutors, Student.~Tutors, Teacher, Teacher.*Tutors, Teacher.Tutors, Teacher.^Tutors, Teacher.~Tutors, Teaches.Class, Teaches.Class.~Teaches, Tutors.Person, Tutors.Person.*Tutors, Tutors.Person.Tutors, Tutors.Person.^Tutors, Tutors.Person.~Tutors, Tutors.Student, Tutors.Student.*Tutors, Tutors.Student.Tutors, Tutors.Student.^Tutors, Tutors.Student.~Tutors, Tutors.Teacher, Tutors.Teacher.*Tutors, Tutors.Teacher.Tutors, Tutors.Teacher.^Tutors, Tutors.Teacher.~Tutors, Tutors.Teaches.Class, ^Tutors.Person, ^Tutors.Person.*Tutors, ^Tutors.Person.Tutors, ^Tutors.Person.^Tutors, ^Tutors.Person.~Tutors, ^Tutors.Student, ^Tutors.Student.*Tutors, ^Tutors.Student.Tutors, ^Tutors.Student.^Tutors, ^Tutors.Student.~Tutors, ^Tutors.Teacher, ^Tutors.Teacher.*Tutors, ^Tutors.Teacher.Tutors, ^Tutors.Teacher.^Tutors, ^Tutors.Teacher.~Tutors, ^Tutors.Teaches.Class, univ.Groups.Group, ~Teaches.Person.*Tutors, ~Teaches.Person.Teaches, ~Teaches.Person.Tutors, ~Teaches.Person.^Tutors, ~Teaches.Person.~Tutors, ~Teaches.Student.*Tutors, ~Teaches.Student.Teaches, ~Teaches.Student.Tutors, ~Teaches.Student.^Tutors, ~Teaches.Student.~Tutors, ~Teaches.Teacher.*Tutors, ~Teaches.Teacher.Teaches, ~Teaches.Teacher.Tutors, ~Teaches.Teacher.^Tutors, ~Teaches.Teacher.~Tutors, ~Tutors.Person, ~Tutors.Person.*Tutors, ~Tutors.Person.Tutors, ~Tutors.Person.^Tutors, ~Tutors.Person.~Tutors, ~Tutors.Student, ~Tutors.Student.*Tutors, ~Tutors.Student.Tutors, ~Tutors.Student.^Tutors, ~Tutors.Student.~Tutors, ~Tutors.Teacher, ~Tutors.Teacher.*Tutors, ~Tutors.Teacher.Tutors, ~Tutors.Teacher.^Tutors, ~Tutors.Teacher.~Tutors, ~Tutors.Teaches.Class, *Tutors.Teaches.univ, *Tutors.univ.Tutors, Teaches.univ, Teaches.univ.*Tutors, Teaches.univ.Tutors, Teaches.univ.^Tutors, Teaches.univ.~Teaches, Teaches.univ.~Tutors, Tutors.Teaches.univ, Tutors.univ, Tutors.univ.*Tutors, Tutors.univ.Tutors, Tutors.univ.^Tutors, Tutors.univ.~Teaches, Tutor

Bad pipe message: %s [b'\xe1ZO\xe39\xde\xd7\xca\xdf<R\x11G\x0f~\xf20_\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00']
Bad pipe message: %s [b'\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00']
Bad pipe message: %s [b'\xbb_\xe3\x9ex\x1b\xe7, \r\x07\x1a\x98\\\x86!\xe4J\x00\x01|\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00\x9f\x00\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xba\x00\xbb\x00\xbc\x00\xbd\x00\xbe\x00\x